# Sternhaufen mit Algorithmen finden

Im ersten Notebook hast du den Haufen von Hand gesucht. Jetzt übernehmen **Clustering-
Algorithmen** dieselbe Aufgabe — dieselben Verfahren, die auch in der professionellen
Astronomie eingesetzt werden.

Ein Clustering-Algorithmus bekommt nur Zahlen und sucht darin Bereiche, in denen die Punkte
dichter liegen als anderswo. Er weiß nichts über Sterne. **Er sieht immer nur die Spalten, die
du ihm gibst** — und genau das probierst du hier aus. Du gibst ihm dreimal etwas anderes:

1. **`ra`, `dec`** — nur der Ort am Himmel, also das, was ein Bild vom Himmel zeigt
2. **`X`, `Y`, `Z`** — die wirkliche Position im Raum, mit der Entfernung als dritter Achse
3. **`X`, `Y`, `Z`, `U`, `V`, `W`** — der volle Phasenraum, Position und Bewegung zusammen

Nach jedem Durchlauf vergleichst du. Es geht nicht darum, den einen richtigen Durchlauf zu
finden, sondern zu sehen, was jeder von ihnen erkennen kann und was nicht.

Du lernst drei Verfahren kennen:

| Verfahren | Idee | Wichtigster Regler |
| --- | --- | --- |
| **HDBSCAN** | sucht dichte Bereiche, erkennt die Anzahl der Haufen selbst | `min_cluster_size` |
| **DBSCAN** | dichte Bereiche bei *einem* festen Suchradius | `eps` |
| **GMM** | legt eine vorgegebene Anzahl von Verteilungen über die Daten | `n_cluster` |

HDBSCAN und DBSCAN dürfen Sterne auch **keinem** Haufen zuordnen — die bekommen das Label
`-1`. Das GMM muss jeden Stern einsortieren.

Die Regler in der letzten Spalte kannst du weglassen. `find_one_cluster` bestimmt sie dann aus
der Menge und den Abständen deiner Daten und schreibt die gewählten Werte in die Ausgabe. Das
ist ein brauchbarer Startpunkt, keine richtige Antwort: Der eigentliche Sinn dieses Notebooks
ist, selbst an den Reglern zu drehen und zu sehen, wie stark das Ergebnis davon abhängt.

## Vorbereitung

Wir laden die Tabelle, die du am Ende des ersten Notebooks gespeichert hast. Sie enthält die
Gaia-Messwerte, die umgerechneten Spalten `X`, `Y`, `Z`, `U`, `V`, `W` und deine Handauswahl in
`auswahl_gesamt` — die brauchst du später zum Vergleichen.

In [ ]:
%matplotlib widget

from stellar_cluster_finder import (
    find_one_cluster,
    load_parquet,
    plot_3d,
    plot_and_save,
    save_parquet,
)

sterne = load_parquet("../data/meine_auswahl.parquet")
print(f"{len(sterne)} Sterne geladen")

Jede Darstellung erzeugt ein **neues** Bild — du musst nichts aufräumen, damit das nächste
richtig aussieht. Weil du hier aber viel wiederholst, sammeln sich die Bilder mit der Zeit an.
Ab 20 offenen Bildern warnt matplotlib. Dann räumst du so auf:

```python
import matplotlib.pyplot as plt

plt.close("all")
```

## 1. Nur der Ort am Himmel: `ra` und `dec`

Zum Aufwärmen bekommt der Algorithmus dieselben zwei Spalten, mit denen du im ersten Notebook
angefangen hast: den Ort am Himmel. Mehr weiß er nicht — weder wie weit die Sterne entfernt
sind noch wie sie sich bewegen.

**Der Ablauf ist immer derselbe wie beim Auswählen von Hand:** erst ansehen, dann suchen
lassen, dann noch einmal ansehen. Also zuerst das Bild ohne jede Farbe.

In [ ]:
plot_and_save(
    sterne,
    "ra",
    "dec",
    title="Die Sterne am Himmel, noch ohne Gruppen",
    xlabel="Rektaszension (Grad)",
    ylabel="Deklination (Grad)",
)

Präge dir ein, wo du selbst eine Gruppe vermuten würdest. Jetzt lässt du den Algorithmus
suchen.

Jeder Durchlauf legt eine neue Spalte an. Über `cluster_label_column` gibst du ihr einen Namen —
**wichtig**, denn sonst überschreibt der nächste Durchlauf den vorherigen.

In [ ]:
find_one_cluster(
    sterne,
    columns=["ra", "dec"],
    # min_cluster_size lassen wir zunächst weg: die Funktion wählt dann selbst einen Wert,
    # der zur Größe deiner Daten passt, und schreibt ihn in die Ausgabe. Gleich drehst du daran.
    mode="HDBSCAN",
    cluster_label_column="himmel",
)

anzahl = (sterne["himmel_HDBSCAN"] >= 0).sum()
print(f"{anzahl} Sterne einem Haufen zugeordnet, {len(sterne) - anzahl} als Rauschen (-1)")

Und dasselbe Bild noch einmal, diesmal eingefärbt nach dem Ergebnis.

In [ ]:
plot_and_save(
    sterne,
    "ra",
    "dec",
    title="Was HDBSCAN am Himmel findet",
    xlabel="Rektaszension (Grad)",
    ylabel="Deklination (Grad)",
    color_col="himmel_HDBSCAN",
)

### Ein Ergebnis, das nach nichts aussieht

Sieh dir die Ausgabe der letzten Zelle genau an. Sehr wahrscheinlich steht dort eine Warnung:
Die größte Gruppe enthält mehr als die Hälfte aller Sterne. Eine Gruppe dieser Größe ist das
Feld selbst und kein Sternhaufen.

Das ist weder dein Fehler noch der des Algorithmus. Am Himmel liegt der Haufen als leichte
Verdichtung **vor** den übrigen Sternen, nicht neben ihnen: Zwischen Haufen und Feld gibt es
keine Lücke — und eine Lücke ist genau das, wonach ein dichtebasiertes Verfahren sucht.

Sieh dir noch einmal das erste, farblose Bild an. Hättest du allein daraus einen Haufen
abgegrenzt? Und woran hast du im ersten Notebook den Haufen tatsächlich erkannt — war es der
Ort am Himmel?

Der Algorithmus hat hier also nicht versagt, sondern gezeigt, dass in diesen beiden Spalten
etwas fehlt. Im nächsten Abschnitt bekommt er eine Achse dazu.

> **Drehe am Regler.** Über der Ausgabe steht, welches `min_cluster_size` die Funktion
> gewählt hat. Nimm diesen Wert als Ausgangspunkt, setze ihn in der Zelle oben von Hand ein
> und probiere der Reihe nach ein Fünftel, das Doppelte und das Zehnfache davon. Beobachte:
> - Ab wann findet der Algorithmus gar nichts mehr?
> - Ab wann wird alles zu einem einzigen großen Haufen?
> - Bei welchem Wert sieht das Ergebnis für dich am sinnvollsten aus?
>
> Wenn die gefundene Gruppe *genau* so groß ist wie dein `min_cluster_size`, warnt die
> Funktion dich. Diese Zahl kommt dann von deiner Einstellung und nicht aus den Daten.
>
> Die Warnung über die überschriebene Spalte beim erneuten Ausführen ist dagegen harmlos —
> sie sagt dir nur, dass das alte Ergebnis ersetzt wird.

### Dieselben Daten, andere Verfahren

Jetzt bekommen alle drei Verfahren genau dieselben zwei Spalten. Jedes legt seine eigene
Ergebnisspalte an, sodass du sie nebeneinander vergleichen kannst.

In [ ]:
for verfahren in ["HDBSCAN", "DBSCAN", "GMM"]:
    find_one_cluster(
        sterne,
        columns=["ra", "dec"],
        # min_cluster_size (HDBSCAN) sowie eps und min_samples (DBSCAN) lassen wir offen und
        # von der Funktion aus den Daten bestimmen. Die gewählten Werte stehen in der Ausgabe.
        n_cluster=2,  # für GMM  <-- diesen Regler musst du selbst setzen
        mode=verfahren,
        cluster_label_column="vergleich",
    )
    spalte = f"vergleich_{verfahren}"
    gefunden = sterne.loc[sterne[spalte] >= 0, spalte].nunique()
    print(f"{verfahren:8s} {gefunden} Haufen, {(sterne[spalte] >= 0).sum():5d} Sterne zugeordnet")

In [ ]:
for verfahren in ["HDBSCAN", "DBSCAN", "GMM"]:
    plot_and_save(
        sterne,
        "ra",
        "dec",
        title=f"{verfahren} am Himmel",
        xlabel="Rektaszension (Grad)",
        ylabel="Deklination (Grad)",
        color_col=f"vergleich_{verfahren}",
    )

> **Vergleiche die drei Bilder.** Wo sind sich die Verfahren einig, wo nicht? Beachte
> besonders, dass das GMM keine grauen Punkte hat: Es *muss* jeden Stern einsortieren, auch
> wenn er offensichtlich nirgends dazugehört.

> **Dreh am Regler von DBSCAN.** `eps` ist ein Radius in denselben Einheiten wie die Daten,
> hier also in Grad am Himmel: DBSCAN zählt für jeden Stern, wie viele andere innerhalb dieses
> Radius liegen. Es gibt keinen Wert, der in allen Spalten dasselbe bedeutet — ein halbes Grad
> am Himmel und ein halbes Parsec im Raum sind völlig verschiedene Dinge. Deshalb misst die
> Funktion `eps` aus den Abständen der Sterne selbst, wenn du keinen Wert angibst, und gibt
> ihn aus.
>
> Setze nun `eps=...` von Hand in die Zelle oben ein und probiere den ausgegebenen Wert, sowie
> die Hälfte, das Doppelte und das Zehnfache davon:
> - Was passiert bei kleinem `eps` — warum zerfällt der Haufen in viele Stücke?
> - Was passiert bei großem `eps` — wann verschmilzt alles zu einer einzigen Gruppe?
> - Gibt es einen Bereich, in dem sich das Ergebnis kaum ändert? Wie sicher wärst du dir bei
>   einem Wert aus diesem Bereich, verglichen mit einem am Rand?
>
> HDBSCAN hat diesen Regler bewusst nicht: Es probiert alle Radien durch und behält, was am
> stabilsten bleibt. Genau das ist das „H" im Namen.

## 2. Die wirkliche Position: `X`, `Y`, `Z`

Am Himmel steht alles nebeneinander, was zufällig in dieselbe Richtung fällt — ein naher und
ein sehr ferner Stern sehen dort gleich aus. Jetzt bekommt der Algorithmus die Entfernung als
dritte Achse dazu und arbeitet damit zum ersten Mal mit wirklichen Abständen im Raum.

Wieder zuerst ansehen, diesmal in 3D. Du kannst die Darstellung **mit der Maus drehen**.

In [ ]:
plot_3d(
    sterne,
    "X",
    "Y",
    "Z",
    title="Die Sterne im Raum, noch ohne Gruppen",
    xlabel="X (pc)",
    ylabel="Y (pc)",
    zlabel="Z (pc)",
)

In [ ]:
find_one_cluster(
    sterne,
    columns=["X", "Y", "Z"],
    mode="HDBSCAN",
    cluster_label_column="position_3d",
)

for spalte in ["himmel_HDBSCAN", "position_3d_HDBSCAN"]:
    print(f"{spalte:22s} {(sterne[spalte] >= 0).sum():5d} Sterne zugeordnet")

In [ ]:
plot_3d(
    sterne,
    "X",
    "Y",
    "Z",
    title="Was HDBSCAN im Raum findet",
    xlabel="X (pc)",
    ylabel="Y (pc)",
    zlabel="Z (pc)",
    color_col="position_3d_HDBSCAN",
)

> **Drehe die Darstellung** und suche Sterne, die am Himmel noch mitten im Haufen lagen, jetzt
> aber als Rauschen (grau) markiert sind. Was hat die Entfernung als dritte Achse verraten,
> das im Bild vom Himmel nicht zu sehen war?
>
> Die zweite Warnung betrifft die Sterne ohne brauchbare Parallaxe: Sie haben keine Position im
> Raum und werden deshalb als Rauschen einsortiert.

## 3. Der volle Phasenraum: `X`, `Y`, `Z`, `U`, `V`, `W`

Zum Schluss bekommt der Algorithmus alles: **wo** die Sterne stehen und **wie** sie sich
bewegen, alle sechs Spalten zusammen. Diese sechs Zahlen nennt man den *Phasenraum*.

Das klingt nach der besten aller Möglichkeiten — mehr Information kann ja nicht schaden. Hier
lauert allerdings eine Falle, die du gleich selbst sehen wirst.

In [ ]:
find_one_cluster(
    sterne,
    columns=["X", "Y", "Z", "U", "V", "W"],
    mode="HDBSCAN",
    cluster_label_column="phasenraum",
)

for spalte in ["himmel_HDBSCAN", "position_3d_HDBSCAN", "phasenraum_HDBSCAN"]:
    print(f"{spalte:22s} {(sterne[spalte] >= 0).sum():5d} Sterne zugeordnet")

In [ ]:
plot_3d(
    sterne,
    "X",
    "Y",
    "Z",
    title="Was HDBSCAN im vollen Phasenraum findet",
    xlabel="X (pc)",
    ylabel="Y (pc)",
    zlabel="Z (pc)",
    color_col="phasenraum_HDBSCAN",
    alpha=0.8,
)

In [ ]:
print(sterne[["X", "Y", "Z", "U", "V", "W"]].describe().loc[["min", "max", "std"]].round(1))

**Sieh dir die Zahlen oben genau an.** Die Positionen stehen in Parsec und gehen über Hunderte,
die Geschwindigkeiten in km/s nur über einige Dutzend. Der Algorithmus kennt aber weder Parsec
noch km/s — für ihn sind es nur Zahlen. Die *größeren* Zahlen bestimmen deshalb fast allein,
welche Sterne er für „nah beieinander" hält.

> **Frage:** Welche der sechs Spalten dominiert hier das Ergebnis? Und ist das physikalisch
> sinnvoll, wo doch die gemeinsame Bewegung das stärkste Kennzeichen eines Haufens ist?
>
> **Und weiter gedacht:** Wenn die Position das Ergebnis dominiert, was würde der Algorithmus
> wohl finden, wenn er *nur* die Bewegung sähe? Genau das ist der optionale Abschnitt 4.

Zwei Dinge fallen an diesem Durchlauf außerdem auf. Erstens ist die Warnung über die fehlenden
Werte hier viel größer als vorher: Für `U`, `V` und `W` braucht es eine Radialgeschwindigkeit,
und die hat Gaia nur für einen kleinen Teil der Sterne gemessen. Alle übrigen fallen aus diesem
Durchlauf heraus — vergleiche die Zahl der zugeordneten Sterne mit der aus Abschnitt 2.

Zweitens kann es sein, dass die gefundene Gruppe *genau* so groß ist wie `min_cluster_size`,
und die Funktion dich davor warnt. Dann hat HDBSCAN nur die Spitze einer einzelnen Verdichtung
abgeschnitten. Probiere in diesem Fall einen anderen Wert und sieh nach, ob die Gruppe
mitwächst: Eine Zahl, die deiner Einstellung folgt statt den Daten, sagt nichts über den Haufen.

## 4. Optional: nur die Bewegung

*Dieser Abschnitt ist eine Zugabe. Wenn die Zeit knapp ist, überspringe ihn und mach mit
Abschnitt 5 weiter — du kannst jederzeit hierher zurückkommen.*

Bisher hat der Algorithmus in jedem Durchlauf gewusst, **wo** die Sterne stehen. Jetzt nimmst
du ihm das weg und gibst ihm nur noch, **wie** sie sich bewegen. Das ist der schärfste Test:
Ein Haufen ist gemeinsam entstanden und fliegt bis heute gemeinsam durch die Galaxie, während
zufällig benachbarte Sterne in alle Richtungen unterwegs sind.

### 4a. Die Eigenbewegung: `pmra` und `pmdec`

Zurück zu den Rohdaten von Gaia — dieselben zwei Spalten, mit denen du im ersten Notebook
deine dritte Auswahl getroffen hast. Wie immer: erst ansehen.

Ein Hinweis vorweg: Einige wenige sehr schnelle Sterne ziehen beide Achsen weit auseinander,
sodass der Klumpen zu einem Punkt in der Mitte zusammenschrumpft. Mit `xlim` und `ylim`
schneidest du beide Achsen zu und holst ihn dir heran. An der Tabelle ändert das nichts, nur am
Ausschnitt. Ein guter erster Schnitt lässt auf jeder Achse das äußerste Prozent weg; danach
liest du engere Zahlen an den Achsen ab und setzt sie von Hand ein.

In [ ]:
plot_and_save(
    sterne,
    "pmra",
    "pmdec",
    title="Die Eigenbewegungen, noch ohne Gruppen",
    xlabel="Eigenbewegung in RA (mas/Jahr)",
    ylabel="Eigenbewegung in Dec (mas/Jahr)",
    # Die äußersten 1% je Achse weglassen — das sind die wenigen sehr schnellen Sterne, die
    # das Bild auseinanderziehen. Danach engere Zahlen von Hand einsetzen.
    # xlim=(float(sterne["pmra"].quantile(0.01)), float(sterne["pmra"].quantile(0.99))),
    # ylim=(float(sterne["pmdec"].quantile(0.01)), float(sterne["pmdec"].quantile(0.99))),
)

In [ ]:
for verfahren in ["HDBSCAN", "DBSCAN", "GMM"]:
    find_one_cluster(
        sterne,
        columns=["pmra", "pmdec"],
        n_cluster=2,  # für GMM  <-- diesen Regler musst du wieder selbst setzen
        mode=verfahren,
        cluster_label_column="eigenbewegung",
    )
    spalte = f"eigenbewegung_{verfahren}"
    gruppen = sterne.loc[sterne[spalte] >= 0, spalte].nunique()
    print(f"{verfahren:8s} {gruppen} Gruppe(n), {(sterne[spalte] >= 0).sum():5d} Sterne zugeordnet")

In [ ]:
for verfahren in ["HDBSCAN", "DBSCAN", "GMM"]:
    plot_and_save(
        sterne,
        "pmra",
        "pmdec",
        title=f"Was {verfahren} in der Eigenbewegung findet",
        xlabel="Eigenbewegung in RA (mas/Jahr)",
        ylabel="Eigenbewegung in Dec (mas/Jahr)",
        color_col=f"eigenbewegung_{verfahren}",
        # Derselbe Ausschnitt wie oben, dann siehst du die Gruppen genauer.
        # xlim=(float(sterne["pmra"].quantile(0.01)), float(sterne["pmra"].quantile(0.99))),
        # ylim=(float(sterne["pmdec"].quantile(0.01)), float(sterne["pmdec"].quantile(0.99))),
    )

> **Vergleiche die drei Bilder — hier trennen sich die Verfahren.**
>
> Die Eigenbewegungen der Feldsterne bilden keine ordentliche Wolke mit Rand, sondern eine
> breite Verteilung, in der der Haufen als dichter Klumpen sitzt — je nach Haufen mehr oder
> weniger deutlich abgesetzt. HDBSCAN verlangt eine Lücke zwischen Klumpen und Feld; findet es
> keine, gibt es das ganze Feld als eine Gruppe zurück und warnt dich davor. DBSCAN misst
> stattdessen den typischen Abstand zwischen den Sternen und schneidet dort ab — und trennt
> den Klumpen deshalb auch dann heraus, wenn keine echte Lücke da ist.
>
> Das GMM geht die Sache grundsätzlich anders an: Es legt `n_cluster` Glockenkurven über die
> Daten und gibt jedem Stern die Kurve, zu der er am besten passt. Das unterstellt, dass die
> Daten aus Glockenkurven aufgebaut sind. Sieh dir daraufhin an, wo die Grenze zwischen den
> GMM-Gruppen verläuft — schneidet sie den Klumpen heraus, oder teilt sie die breite Verteilung
> der Feldsterne in zwei Hälften? Und was sagt dir das darüber, ob die Eigenbewegungen der
> Feldsterne die Form haben, die das Verfahren erwartet?
>
> - Welches Verfahren hat bei *deinem* Haufen die überzeugendere Gruppe gefunden?
> - Sieh dir die mittlere Entfernung der gefundenen Sterne an. Liegt sie bei deiner Spitze aus
>   dem ersten Notebook, oder viel weiter draußen — bei den Feldsternen?
> - Beim GMM gibt es keinen Wert, den die Funktion für dich aus den Daten bestimmen könnte:
>   `n_cluster` musst du selbst raten, während `min_cluster_size` und `eps` gemessen werden.
>   Probiere `n_cluster=3` und `n_cluster=5`. Wird die Gruppe auf dem Haufen dadurch sauberer,
>   oder wird nur das Feld feiner zerschnitten?
> - In Abschnitt 2 war es umgekehrt: Dort hat HDBSCAN in `X`, `Y`, `Z` gut funktioniert. Kein
>   Verfahren ist überall das beste, und genau deshalb probiert man mehrere aus.

### 4b. Die wirkliche Geschwindigkeit: `U`, `V`, `W`

Die Eigenbewegung ist ein *Winkel* pro Jahr: Zwei Sterne mit derselben Eigenbewegung sind
verschieden schnell unterwegs, wenn sie verschieden weit weg sind. `U`, `V`, `W` sind die
daraus berechneten echten Geschwindigkeiten in km/s. Findet der Algorithmus damit denselben
Haufen?

In [ ]:
find_one_cluster(
    sterne,
    columns=["U", "V", "W"],
    mode="HDBSCAN",
    cluster_label_column="geschwindigkeit_3d",
)

for spalte in ["eigenbewegung_HDBSCAN", "eigenbewegung_DBSCAN", "geschwindigkeit_3d_HDBSCAN"]:
    print(f"{spalte:28s} {(sterne[spalte] >= 0).sum():5d} Sterne zugeordnet")

Und jetzt der aufschlussreichste Blick des ganzen Notebooks: **gruppiert wurde nach der
Bewegung, dargestellt ist die Position.** Wo die Sterne stehen, hat bei dieser Gruppierung
überhaupt keine Rolle gespielt.

In [ ]:
plot_3d(
    sterne,
    "X",
    "Y",
    "Z",
    title="Nach der Geschwindigkeit gruppiert, im Raum dargestellt",
    xlabel="X (pc)",
    ylabel="Y (pc)",
    zlabel="Z (pc)",
    color_col="geschwindigkeit_3d_HDBSCAN",
)

> **Drehe die Darstellung.** Wenn die farbigen Sterne auch im Raum zusammenliegen, obwohl der
> Algorithmus die Position gar nicht kannte, sind sich zwei völlig unabhängige Messungen einig.
> Das ist ein deutlich stärkeres Argument für einen echten Sternhaufen als jede Messung für
> sich allein — und es ist der Grund, warum in der Astronomie so viel Aufwand getrieben wird,
> dieselbe Sache auf zwei verschiedene Arten zu messen.

## 5. Algorithmus gegen Handauswahl

Zum Schluss der Vergleich mit deiner eigenen Auswahl aus dem ersten Notebook. Trage unten die
Spalte des Durchlaufs ein, der dich am meisten überzeugt hat.

In [ ]:
meine = sterne["auswahl_gesamt"]
algorithmus = sterne["position_3d_HDBSCAN"] >= 0  # <-- hier eine andere Spalte einsetzen

print(f"nur ich:             {(meine & ~algorithmus).sum():5d} Sterne")
print(f"nur der Algorithmus: {(~meine & algorithmus).sum():5d} Sterne")
print(f"beide:               {(meine & algorithmus).sum():5d} Sterne")

In [ ]:
sterne["vergleich_hand_algorithmus"] = "keiner von beiden"
sterne.loc[meine & ~algorithmus, "vergleich_hand_algorithmus"] = "nur ich"
sterne.loc[~meine & algorithmus, "vergleich_hand_algorithmus"] = "nur der Algorithmus"
sterne.loc[meine & algorithmus, "vergleich_hand_algorithmus"] = "beide"

plot_and_save(
    sterne,
    "ra",
    "dec",
    title="Wo sind wir uns einig?",
    xlabel="Rektaszension (Grad)",
    ylabel="Deklination (Grad)",
    color_col="vergleich_hand_algorithmus",
)

Deine Handauswahl hat drei Messgrößen kombiniert — Entfernung, Ort am Himmel und
Eigenbewegung. Jeder einzelne Durchlauf des Algorithmus hat weniger gesehen als du. Behalte das
im Kopf, wenn ihr euch uneinig seid.

### Ergebnis sichern

Damit du im optionalen dritten Notebook mit deinen Cluster-Ergebnissen weiterarbeiten kannst,
speicherst du die Tabelle mit allen Label-Spalten noch einmal ab.

In [ ]:
save_parquet(sterne, "../data/haufen_mit_clustern.parquet")

## Fragen zum Nachdenken

- Bei welchen Sternen seid ihr euch uneinig? Sieh dir einige davon genauer an — wer hat deiner
  Meinung nach recht?
- Welcher Datenraum hat den Haufen am saubersten herausgetrennt: der Ort am Himmel, die
  Position im Raum oder der volle Phasenraum? Hättest du das vorher erwartet?
- `eps` bei DBSCAN und `min_cluster_size` bei HDBSCAN wirken beide auf die Größe der gefundenen
  Gruppen. Worin unterscheiden sie sich trotzdem?
- Der Phasenraum enthält die meiste Information. Warum liefert „mehr Information" trotzdem
  nicht automatisch das beste Ergebnis?
- HDBSCAN und DBSCAN dürfen Sterne als Rauschen markieren, das GMM nicht. Wann ist welches
  Verhalten das bessere?
- Alle drei Verfahren haben Regler, die das Ergebnis stark verändern. Woher willst du wissen,
  welche Einstellung „richtig" ist — wenn du die Antwort doch gar nicht kennst?

Diese letzte Frage hat keine einfache Antwort. Genau deshalb schauen sich Forschende ihre
Daten immer auch von Hand an, so wie du im ersten Notebook.

---

**Weiter geht es im dritten Notebook:** Dort geht es nicht mehr darum, *welche* Sterne
zusammengehören, sondern wie *alt* sie sind.
